In [ ]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.4/418.4 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 125.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22

In [ ]:
import json
from tqdm import tqdm
from unsloth import FastLanguageModel
import torch
from transformers import StoppingCriteria, StoppingCriteriaList

MODEL_PATH = "/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed"
INPUT_DATASET = "/content/data.jsonl"
OUTPUT_FILE = "model_outputs.jsonl"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

class StopOnCodeEnd(StoppingCriteria):
    def __call__(self, input_ids, scores, **kwargs):
        decoded = tokenizer.decode(input_ids[0][-5:])
        return "</code>" in decoded

stop_criteria = StoppingCriteriaList([StopOnCodeEnd()])

with open(INPUT_DATASET, 'r') as f:
    dataset = [json.loads(line) for line in f]

outputs = []
for item in tqdm(dataset, desc="Generating"):
    # Fix: Get the question using 'prompt' or 'instruction'
    problem_text = item.get('prompt', item.get('instruction', ''))

    prompt = f"""### Instruction:
Solve this competitive programming problem.
1. Use standard I/O (sys.stdin.read() or input()).
2. Output logic must be exactly: <tags>...</tags><think>...</think><code>...</code>.
3. NO example usage or hardcoded variables.

Problem: {problem_text}

### Response:
<tags>"""

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    with torch.no_grad():
        generate_ids = model.generate(
            **inputs,
            max_new_tokens = 1500,
            temperature = 0.1,
            top_p = 0.9,
            stopping_criteria = stop_criteria,
            use_cache = True
        )

    full_response = tokenizer.decode(generate_ids[0], skip_special_tokens=True)
    prediction = full_response.split("### Response:")[-1].strip()

    outputs.append({
        "id": item["id"],
        "prediction": prediction
    })

with open(OUTPUT_FILE, 'w') as f:
    for out in outputs:
        f.write(json.dumps(out) + "\n")

==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth: Will load /content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed as a legacy tokenizer.
Generating: 100%|██████████| 100/100 [09:54<00:00,  5.94s/it]


In [ ]:
import json
import subprocess
import sys
import os
import re
import tempfile
from tqdm import tqdm

# --- CONFIGURATION ---
MODEL_SOLUTIONS_FILE = "/content/drive/MyDrive/SLM_DATA/stage2_FINAL_EVAL_MATCHED.jsonl"
GROUND_TRUTH_FILE = "/content/drive/MyDrive/SLM_DATA/test_suite_200_FINAL.jsonl"

def fuzzy_extract(text, start_tag, end_tags):
    """Extracts content starting after start_tag until any of the end_tags or end of string."""
    text_lower = text.lower()
    start_tag = start_tag.lower()

    start_idx = text_lower.find(start_tag)
    if start_idx == -1: return ""

    content_start = start_idx + len(start_tag)
    end_idx = len(text)

    for tag in end_tags:
        pos = text_lower.find(tag.lower(), content_start)
        if pos != -1 and pos < end_idx:
            end_idx = pos

    return text[content_start:end_idx].strip()

def run_test_sandbox(code, input_data):
    """Executes code by piping input_data into STDIN."""
    with tempfile.NamedTemporaryFile(suffix='.py', delete=False, mode='w') as tmp:
        tmp.write(code)
        tmp_path = tmp.name
    try:
        proc = subprocess.run(
            [sys.executable, tmp_path],
            input=str(input_data),
            capture_output=True, text=True, timeout=3
        )
        if proc.returncode != 0: return None, "RE"
        return proc.stdout.strip(), "SUCCESS"
    except subprocess.TimeoutExpired:
        return None, "TLE"
    except Exception:
        return None, "ERROR"
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

def calculate_all_metrics():
    # 1. Load Data
    with open(MODEL_SOLUTIONS_FILE, 'r') as f:
        results = [json.loads(line) for line in f]
    with open(GROUND_TRUTH_FILE, 'r') as f:
        # Match IDs as strings to handle 'vfc_...'
        truth = {str(json.loads(line)['id']): json.loads(line) for line in f}

    # 2. Initialize Counters
    stats = {"passed": 0, "far_count": 0, "total_density": 0, "executed": 0}
    total = len(results)

    print(f"🚀 Starting Unified Evaluation on {total} questions...")

    for res in tqdm(results):
        q_id = str(res['id'])
        prediction = res['prediction']

        # --- A. PARSING & FORMAT METRICS ---
        tags_txt = fuzzy_extract(prediction, "<tags>", ["<think>", "<code>"])
        think_txt = fuzzy_extract(prediction, "<think>", ["<code>", "</think>"])
        code_txt = fuzzy_extract(prediction, "<code>", ["</code>"])

        # FAR: Check for presence of all structural markers
        if "<tags>" in prediction.lower() and "<think>" in prediction.lower() and "<code>" in prediction.lower():
            stats["far_count"] += 1

        # Think Density: (Reasoning content) / (Total output length)
        reasoning_len = len(tags_txt) + len(think_txt)
        if len(prediction) > 0:
            stats["total_density"] += (reasoning_len / len(prediction))

        # --- B. FUNCTIONAL METRICS (Pass@1) ---
        if q_id in truth:
            if not code_txt:
                continue

            stats["executed"] += 1
            test_cases = truth[q_id].get('test_cases', [])
            all_passed = True

            for tc in test_cases:
                actual, status = run_test_sandbox(code_txt, tc['input'])
                if status != "SUCCESS" or actual != str(tc['output']).strip():
                    all_passed = False
                    break

            if all_passed and test_cases:
                stats["passed"] += 1

    # --- 3. FINAL REPORT ---
    pass_rate = (stats["passed"] / total) * 100
    far_rate = (stats["far_count"] / total) * 100
    avg_density = (stats["total_density"] / total) * 100

    print("\n" + "="*45)
    print("🏆 THE SMALL CODER: UNIFIED METRICS REPORT")
    print("="*45)
    print(f"✅ Pass@1 Accuracy:         {pass_rate:>7.2f}%")
    print(f"🧠 Think Density:           {avg_density:>7.2f}%")
    print(f"🏗️ Format Adherence (FAR):  {far_rate:>7.2f}%")
    print("-" * 45)
    print(f"Total Questions: {total}")
    print(f"Successfully Passed: {stats['passed']}")
    print(f"Codes Executed: {stats['executed']}")
    print("="*45)

if __name__ == "__main__":
    calculate_all_metrics()

🚀 Starting Unified Evaluation on 200 questions...


100%|██████████| 200/200 [00:41<00:00,  4.83it/s]


🏆 THE SMALL CODER: UNIFIED METRICS REPORT
✅ Pass@1 Accuracy:            8.00%
🧠 Think Density:             62.63%
🏗️ Format Adherence (FAR):    94.50%
---------------------------------------------
Total Questions: 200
Successfully Passed: 16
Codes Executed: 196


In [ ]:
import json
from tqdm import tqdm
from unsloth import FastLanguageModel
import torch

# Load the 100 questions
with open("/content/data.jsonl", "r") as f:
    dataset = [json.loads(line) for line in f]

outputs = []
for item in tqdm(dataset, desc="Generating Answers"):
    prob_text = item.get('prompt', '')

    # Using your Milestone 2 Prompt
    prompt = f"""### Instruction:
Solve this competitive programming problem.
1. Use standard I/O (sys.stdin.read() or input()).
2. Output logic must be exactly: <tags>...</tags><think>...</think><code>...</code>.

Problem: {prob_text}

### Response:
<tags>"""

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    with torch.no_grad():
        generate_ids = model.generate(
            **inputs,
            max_new_tokens=1500,
            temperature=0.1,
            use_cache=True
        )

    response = tokenizer.decode(generate_ids[0], skip_special_tokens=True).split("### Response:")[-1].strip()

    outputs.append({
        "id": item["id"], # This will be 26, 27, etc.
        "prediction": response
    })

# Save to a fresh file
with open("stage2_100_outputs.jsonl", "w") as f:
    for out in outputs:
        f.write(json.dumps(out) + "\n")

Generating Answers: 100%|██████████| 100/100 [10:46<00:00,  6.46s/it]


In [ ]:
# (Run this after the generation above finishes)
def run_final_judge():
    with open("stage2_100_outputs.jsonl", "r") as f:
        results = [json.loads(line) for line in f]
    with open("/content/data.jsonl", "r") as f:
        truth = {str(json.loads(line)['id']): json.loads(line) for line in f}

    wa_ids = []
    passed = 0

    for res in tqdm(results, desc="Judging"):
        q_id = str(res['id'])
        # Extract code (using the fuzzy logic we perfected earlier)
        code_match = re.search(r'<code>(.*?)</code>', res['prediction'], re.DOTALL | re.IGNORECASE)
        code_txt = code_match.group(1).strip() if code_match else ""

        if q_id in truth and code_txt:
            is_correct = True
            for tc in truth[q_id]['test_cases']:
                # Run your sandbox check here
                actual, status = run_test_sandbox(code_txt, tc['input'])
                if actual != str(tc['output']).strip():
                    is_correct = False
                    break
            if is_correct: passed += 1
            else: wa_ids.append(q_id)
        else:
            wa_ids.append(q_id) # Count as failure if no code or no ID

    # SAVE THE FAILED IDs
    with open("wa_ids_for_rag.json", "w") as f:
        json.dump(wa_ids, f)

    print(f"\n✅ Done! Pass@1: {passed}%")
    print(f"Saved {len(wa_ids)} failed IDs to wa_ids_for_rag.json")

run_final_judge()

Judging: 100%|██████████| 100/100 [00:07<00:00, 12.84it/s]


✅ Done! Pass@1: 8%
Saved 92 failed IDs to wa_ids_for_rag.json


In [ ]:
import json
from collections import Counter
import os

# Paths
GROUND_TRUTH = "/content/data.jsonl"
FAILED_IDS_PATH = "wa_ids_for_rag.json"

def generate_topic_report():
    if not os.path.exists(FAILED_IDS_PATH):
        print(f"❌ Error: {FAILED_IDS_PATH} not found. Ensure the judge script ran successfully.")
        return

    # 1. Load the Truth data (map IDs to Topics)
    with open(GROUND_TRUTH, 'r') as f:
        # We store as strings to ensure matching with the JSON list
        truth_data = {str(json.loads(line)['id']): json.loads(line) for line in f}

    # 2. Load the Failed IDs
    with open(FAILED_IDS_PATH, 'r') as f:
        wa_ids = [str(i) for i in json.load(f)]

    if not wa_ids:
        print("🎉 Zero failures found! Your model passed everything.")
        return

    # 3. Categorize Failures
    failed_topics = []
    total_found = 0

    for q_id in wa_ids:
        if q_id in truth_data:
            topic = truth_data[q_id].get('topic', 'Unknown')
            failed_topics.append(topic)
            total_found += 1
        else:
            # Debugging if IDs still don't match
            pass

    # 4. Print the Heatmap
    report = Counter(failed_topics)

    print("\n" + "="*45)
    print("🔥 TOPIC FAILURE HEATMAP (100 Qs)")
    print("="*45)
    print(f"{'ALGORITHMIC TOPIC':<25} | {'FAILURES':<10}")
    print("-" * 45)

    for topic, count in report.most_common():
        # Visual bar for easy scanning
        bar = "█" * count
        print(f"{topic:<25} | {count:<10} {bar}")

    print("-" * 45)
    print(f"Total Failures Mapped: {total_found} / {len(wa_ids)}")
    print("="*45)

generate_topic_report()


🔥 TOPIC FAILURE HEATMAP (100 Qs)
ALGORITHMIC TOPIC         | FAILURES  
---------------------------------------------
Array                     | 14         ██████████████
Two Pointer               | 9          █████████
Binary Tree               | 9          █████████
Hash Map                  | 8          ████████
Binary Search             | 8          ████████
Greedy                    | 7          ███████
Linked List               | 6          ██████
Bit Manipulation          | 5          █████
Math                      | 5          █████
Stack                     | 4          ████
String                    | 3          ███
Tree                      | 3          ███
Dynamic Programming       | 2          ██
Set                       | 2          ██
Simulation                | 2          ██
Sorting                   | 2          ██
Hash Table                | 1          █
Sliding Window            | 1          █
Hash Set                  | 1          █
-----------------------------

In [ ]:
import json

# Targeted Blueprints for your Top Failures
knowledge_base = [
    {
        "topic": "Array",
        "blueprint": "[LOGIC]: Use index-based iteration (for i in range(n)). For in-place changes, use a separate 'write_index'. [PIVOT]: Handle boundary cases (empty array, single element) explicitly before the loop."
    },
    {
        "topic": "Two Pointer",
        "blueprint": "[LOGIC]: Use 'left' and 'right' pointers. [PIVOT]: If searching for a sum or pair, SORT the array first. Move 'left' inward to increase value, 'right' inward to decrease value."
    },
    {
        "topic": "Binary Tree",
        "blueprint": "[LOGIC]: Use recursion (DFS) or a queue (BFS). [PIVOT]: Always define the base case 'if not root: return' first. For path sums, subtract the current node's value from the target in the recursive call."
    },
    {
        "topic": "Hash Map",
        "blueprint": "[LOGIC]: Store 'seen' values as keys. [PIVOT]: Use 'if key in dict' for O(1) lookup to find complements (target - current) or check for duplicates during a single pass."
    },
    {
        "topic": "Binary Search",
        "blueprint": "[LOGIC]: Define [low, high]. While low <= high, calculate mid. [PIVOT]: If searching for an 'insertion point' or 'boundary', use 'low = mid + 1' or 'high = mid - 1' to avoid infinite loops."
    },
    {
        "topic": "Greedy",
        "blueprint": "[LOGIC]: Make the best local choice at each step. [PIVOT]: Usually requires sorting the input based on a specific property (e.g., end times, weights, or ratios) before starting the greedy loop."
    },
    {
        "topic": "Linked List",
        "blueprint": "[LOGIC]: Use a 'dummy' head pointing to the real head. [PIVOT]: Use two pointers (fast/slow) for cycle detection or finding the middle. Always update '.next' pointers carefully to preserve the chain."
    }
]

with open("algo_knowledge_base.jsonl", "w") as f:
    for entry in knowledge_base:
        f.write(json.dumps(entry) + "\n")

print("✅ High-Density Knowledge Base created for top failure topics.")

✅ High-Density Knowledge Base created for top failure topics.


In [ ]:
# 1. Force remove all opentelemetry related folders from the file system
import shutil
import sys
import os

# Find where packages are installed
site_packages = [p for p in sys.path if 'site-packages' in p]
for sp in site_packages:
    otel_path = os.path.join(sp, 'opentelemetry')
    if os.path.exists(otel_path):
        print(f"Removing broken folder: {otel_path}")
        shutil.rmtree(otel_path)

# 2. Uninstall using pip to clean up metadata
!pip uninstall -y opentelemetry-api opentelemetry-sdk opentelemetry-semantic-conventions \
    opentelemetry-proto opentelemetry-exporter-otlp-proto-grpc chromadb

# 3. Install chromadb cleanly (it will pull the correct version of opentelemetry)
!pip install chromadb==0.4.24 sentence-transformers -q

print("\n✅ Cleanup finished! CRITICAL: Restart your runtime now (Runtime > Restart Session).")

Found existing installation: opentelemetry-api 1.38.0
Uninstalling opentelemetry-api-1.38.0:
  Successfully uninstalled opentelemetry-api-1.38.0
Found existing installation: opentelemetry-sdk 1.38.0
Uninstalling opentelemetry-sdk-1.38.0:
  Successfully uninstalled opentelemetry-sdk-1.38.0
Found existing installation: opentelemetry-semantic-conventions 0.59b0
Uninstalling opentelemetry-semantic-conventions-0.59b0:
  Successfully uninstalled opentelemetry-semantic-conventions-0.59b0
Found existing installation: opentelemetry-proto 1.38.0
Uninstalling opentelemetry-proto-1.38.0:
  Successfully uninstalled opentelemetry-proto-1.38.0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.5/525.5 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.4 MB/

In [ ]:
# 1. Force downgrade NumPy to the last 1.x version
!pip install "numpy<2.0.0" -q

# 2. Re-verify chromadb is happy
!pip install chromadb sentence-transformers -q

print("\n✅ NumPy downgraded! CRITICAL: You MUST restart your runtime now.")
print("Go to: Runtime > Restart Session")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 105.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.

In [ ]:
import numpy as np
import chromadb
print(f"NumPy version: {np.__version__}")
client = chromadb.Client()
print("🚀 ChromaDB is finally ready for RAG!")

NumPy version: 1.26.4


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


🚀 ChromaDB is finally ready for RAG!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import torch
from tqdm import tqdm
import chromadb
from sentence_transformers import SentenceTransformer
from unsloth import FastLanguageModel
from transformers import StoppingCriteria, StoppingCriteriaList

# --- 1. SETUP & INITIALIZATION ---
MODEL_PATH = "/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

class StopOnCodeEnd(StoppingCriteria):
    def __call__(self, input_ids, scores, **kwargs):
        decoded = tokenizer.decode(input_ids[0][-5:])
        return "</code>" in decoded

stop_criteria = StoppingCriteriaList([StopOnCodeEnd()])

# Initialize RAG
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="cp_expert_blueprints")

# Index the blueprints
with open("algo_knowledge_base.jsonl", "r") as f:
    kb_data = [json.loads(line) for line in f]

collection.add(
    documents=[item['blueprint'] for item in kb_data],
    metadatas=[{"topic": item['topic']} for item in kb_data],
    ids=[item['topic'] for item in kb_data]
)

# Load the 100 questions
INPUT_DATASET = "/content/data.jsonl"
with open(INPUT_DATASET, 'r') as f:
    dataset = [json.loads(line) for line in f]

# --- 2. REINFORCED RAG INFERENCE LOOP ---
rag_outputs = []

for item in tqdm(dataset, desc="🧠 Reinforced RAG Solving"):
    problem_text = item.get('prompt', '')

    # RETRIEVAL
    results = collection.query(query_texts=[problem_text], n_results=1)
    relevant_blueprint = results['documents'][0][0]

    # THE REINFORCED PROMPT: Explicitly commanding structure and I/O
    prompt = f"""### Instruction:
Solve this competitive programming problem using the Reference Blueprint.
CRITICAL RULES:
1. You MUST follow the structure: <tags>...</tags><think>...</think><code>...</code>.
2. You MUST use Standard I/O (input() or sys.stdin.read()). DO NOT just write a function.

### Reference Blueprint:
{relevant_blueprint}

### Problem:
{problem_text}

### Response:
<tags>"""

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    with torch.no_grad():
        generate_ids = model.generate(
            **inputs,
            max_new_tokens=1500,
            temperature=0.1,
            top_p = 0.9,
            stopping_criteria=stop_criteria,
            use_cache=True
        )

    # Decode and extract only the generated portion
    full_resp = tokenizer.decode(generate_ids[0], skip_special_tokens=True)

    # We split by '### Response:' and keep the content after '<tags>'
    # Since the prompt ends with '<tags>', we prepend it back to the result
    # if it was cut off during decoding.
    prediction = full_resp.split("### Response:")[-1].strip()
    if not prediction.startswith("<tags>"):
        prediction = "<tags>" + prediction

    rag_outputs.append({
        "id": item["id"],
        "prediction": prediction
    })

# --- 3. SAVE RESULTS ---
with open("stage2_RAG_outputs.jsonl", "w") as f:
    for out in rag_outputs:
        f.write(json.dumps(out) + "\n")

print("\n✅ Reinforced RAG Inference Complete!")

==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth: Will load /content/drive/MyDrive/SLM_DATA/DeepSeek-R1-CP-Python-Stage2_Fixed as a legacy tokenizer.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
🧠 Reinforced RAG Solving: 100%|██████████| 100/100 [24:36<00:00, 14.76s/it]


✅ Reinforced RAG Inference Complete!


In [ ]:
import json
import subprocess
import sys
import os
import re
import tempfile
from tqdm import tqdm

MODEL_SOLUTIONS_FILE = "stage2_RAG_outputs.jsonl"
GROUND_TRUTH_FILE = "/content/data.jsonl"

def fuzzy_extract(text, start_tag, end_tags):
    text_lower = text.lower()
    start_tag = start_tag.lower()
    start_idx = text_lower.find(start_tag)
    if start_idx == -1: return ""
    content_start = start_idx + len(start_tag)
    end_idx = len(text)
    for tag in end_tags:
        pos = text_lower.find(tag.lower(), content_start)
        if pos != -1 and pos < end_idx:
            end_idx = pos
    return text[content_start:end_idx].strip()

def run_test_sandbox(code, input_data):
    with tempfile.NamedTemporaryFile(suffix='.py', delete=False, mode='w') as tmp:
        tmp.write(code)
        tmp_path = tmp.name
    try:
        proc = subprocess.run(
            [sys.executable, tmp_path],
            input=str(input_data),
            capture_output=True, text=True, timeout=3
        )
        if proc.returncode != 0: return None, "RE"
        return proc.stdout.strip(), "SUCCESS"
    except subprocess.TimeoutExpired:
        return None, "TLE"
    except Exception:
        return None, "ERROR"
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

def evaluate_metrics():
    with open(MODEL_SOLUTIONS_FILE, 'r') as f:
        results = [json.loads(line) for line in f]
    with open(GROUND_TRUTH_FILE, 'r') as f:
        truth = {str(json.loads(line)['id']): json.loads(line) for line in f}

    stats = {"passed": 0, "far_count": 0, "total_density": 0, "executed": 0}
    total = len(results)

    for res in tqdm(results, desc="Evaluating"):
        q_id = str(res['id'])
        prediction = res['prediction']

        tags_txt = fuzzy_extract(prediction, "<tags>", ["<think>", "<code>"])
        think_txt = fuzzy_extract(prediction, "<think>", ["<code>", "</think>"])
        code_txt = fuzzy_extract(prediction, "<code>", ["</code>"])

        if all(tag in prediction.lower() for tag in ["<tags>", "<think>", "<code>"]):
            stats["far_count"] += 1

        reasoning_len = len(tags_txt) + len(think_txt)
        if len(prediction) > 0:
            stats["total_density"] += (reasoning_len / len(prediction))

        if q_id in truth and code_txt:
            stats["executed"] += 1
            test_cases = truth[q_id].get('test_cases', [])
            all_passed = True
            for tc in test_cases:
                actual, status = run_test_sandbox(code_txt, tc['input'])
                if status != "SUCCESS" or actual != str(tc['output']).strip():
                    all_passed = False
                    break
            if all_passed and test_cases:
                stats["passed"] += 1

    print("\n" + "="*45)
    print("🏆 RAG-AUGMENTED SCOREBOARD")
    print("="*45)
    print(f"✅ Pass@1: { (stats['passed']/total)*100 :>7.2f}%")
    print(f"🧠 Think Density: { (stats['total_density']/total)*100 :>7.2f}%")
    print(f"🏗️ FAR Rate: { (stats['far_count']/total)*100 :>7.2f}%")
    print("-" * 45)
    print(f"Executed: {stats['executed']}/{total}")
    print("="*45)

if __name__ == "__main__":
    evaluate_metrics()

Evaluating: 100%|██████████| 100/100 [00:10<00:00,  9.40it/s]


🏆 RAG-AUGMENTED SCOREBOARD
✅ Pass@1:   11.00%
🧠 Think Density:   62.72%
🏗️ FAR Rate:   96.00%
---------------------------------------------
Executed: 100/100
